## Case Técnico - Engenharia de Dados Júnior (Shared Experience PJ)

## 1. Configuração e Leitura dos Dados

Nesta etapa iniciei a sessão Spark e realizei a leitura da base utilizada no case.

Também defini algumas configurações básicas do ambiente para garantir a execução correta do processamento.

### Importando as bibliotecas necessárias

In [16]:
from __future__ import annotations
from datetime import datetime
from pyspark.sql import SparkSession, DataFrame, functions as F
from pyspark.sql.types import IntegerType, DateType
import pandas as pd

### Configuração da Sessão Spark

In [17]:
spark = (SparkSession.builder
    .appName("etl_super_iam_notebook")
    .config("spark.sql.session.timeZone", "America/Sao_Paulo")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Gerando ID único de execução
exec_id = datetime.now().strftime("%Y%m%d%H%M%S")
print(f"Sessão iniciada | exec_id={exec_id}")

# Leitura da base bruta em Parquet
input_path = "amostra_super_iam.parquet"
df_raw = spark.read.parquet(input_path)

Sessão iniciada | exec_id=20260609005726


## 2. Análise Exploratória e Diagnóstico de Qualidade

Antes de iniciar as transformações, fiz uma análise inicial da base para entender sua estrutura e identificar possíveis problemas de qualidade.

Foram verificadas informações como quantidade de registros, tipos das colunas e exemplos de dados para identificar inconsistências que precisariam ser tratadas nas próximas etapas.

In [18]:
# Contagem total da RAW
linhas_brutas = df_raw.count()
print(f"Volumetria da RAW: {linhas_brutas} registros.")

# Diagnóstico visual do Schema e dos dados
df_raw.printSchema()
df_raw.show(5, truncate=False)

Volumetria da RAW: 5000 registros.
root
 |-- cnpj14: string (nullable = true)
 |-- grupo_segmento: string (nullable = true)
 |-- segmento_detalhado: string (nullable = true)
 |-- modelo_atendimento: string (nullable = true)
 |-- situacao_conta: string (nullable = true)
 |-- situacao_cadastral_receita: string (nullable = true)
 |-- data_abertura_conta: string (nullable = true)
 |-- faixa_tempo_criacao_conta: string (nullable = true)
 |-- data_fundacao_empresa: string (nullable = true)
 |-- faixa_tempo_vida_empresa: string (nullable = true)
 |-- data_inicio_relacionamento_banco: string (nullable = true)
 |-- uf_sigla: string (nullable = true)
 |-- tipo_sociedade: string (nullable = true)
 |-- quantidade_socios: string (nullable = true)
 |-- cod_operador: string (nullable = true)
 |-- tipo_operador: string (nullable = true)
 |-- data_criacao_operador: string (nullable = true)
 |-- faixa_tempo_criacao_operador: string (nullable = true)
 |-- sit_operador: string (nullable = true)
 |-- mot_b

## 3. Limpeza e Padronização

Nesta etapa tratei os principais problemas encontrados na base, como:

- Valores nulos representados por texto ("null", "n/a", etc.)
- Diferenças de preenchimento em campos de resposta (sim, S, true...)
- Campos numéricos armazenados como texto
- Datas em formatos diferentes

O objetivo foi deixar os dados padronizados e mais consistentes antes das transformações de negócio.

In [19]:
def limpar_texto(coluna):
    c = F.lower(F.trim(coluna))
    return F.when(c.isin("null", "none", "n/a", "na", ""), None).otherwise(c)

def limpar_flag(coluna):
    c = F.lower(F.trim(coluna))
    return (F.when(c.isin("sim", "s", "1", "true"), "sim")
             .when(c.isin("nao", "não", "n", "0", "false"), "nao")
             .otherwise(None))

def limpar_inteiro(coluna):
    c = F.regexp_replace(F.trim(coluna), r"\.0+$", "")
    c = F.regexp_replace(c, r"^0+(?=\d)", "")
    return c.cast(IntegerType())

def limpar_data(coluna):
    c = F.trim(coluna)
    c = F.when(c == "9999-12-31", None).otherwise(c)
    return F.coalesce(
        F.to_date(c, "yyyy-MM-dd"),
        F.to_date(c, "yyyy/MM/dd"),
        F.to_date(c, "dd/MM/yyyy")
    )

def limpar_ref_anomes(coluna):
    c = F.trim(coluna)
    return F.coalesce(
        F.to_date(c, "yyyy-MM-dd"),
        F.to_date(c, "yyyy/MM/dd"),
        F.to_date(F.concat(F.substring(c, 1, 4), F.lit("-"), F.substring(c, 5, 2), F.lit("-01")), "yyyy-MM-dd")
    )

def aplicar_limpeza(df: DataFrame) -> DataFrame:
    return (df
        .withColumn("sit_operador",        limpar_texto(F.col("sit_operador")))
        .withColumn("tipo_operador",       limpar_texto(F.col("tipo_operador")))
        .withColumn("situacao_conta",      limpar_texto(F.col("situacao_conta")))
        .withColumn("modelo_atendimento",  limpar_texto(F.col("modelo_atendimento")))
        .withColumn("flag_firmas_e_poderes",          limpar_flag(F.col("flag_firmas_e_poderes")))
        .withColumn("tem_token_mobile_habilitado",    limpar_flag(F.col("tem_token_mobile_habilitado")))
        .withColumn("tem_token_embarcado_habilitado", limpar_flag(F.col("tem_token_embarcado_habilitado")))
        .withColumn("flag_acessou_canal",             limpar_flag(F.col("flag_acessou_canal")))
        .withColumn("flag_acessou_mobile",            limpar_flag(F.col("flag_acessou_mobile")))
        .withColumn("flag_acessou_web",               limpar_flag(F.col("flag_acessou_web")))
        .withColumn("flag_acessou_app_comp",          limpar_flag(F.col("flag_acessou_app_comp")))
        .withColumn("flag_acessou_canal_90d_anteriores", limpar_flag(F.col("flag_acessou_canal_90d_anteriores")))
        .withColumn("cod_operador",                       limpar_inteiro(F.col("cod_operador")))
        .withColumn("quantidade_socios",                  limpar_inteiro(F.col("quantidade_socios")))
        .withColumn("qtd_total_sessoes",                  limpar_inteiro(F.col("qtd_total_sessoes")))
        .withColumn("qtd_operadores_associados_ao_cnpj",  limpar_inteiro(F.col("qtd_operadores_associados_ao_cnpj")))
        .withColumn("qtd_contas_associadas_ao_operador",  limpar_inteiro(F.col("qtd_contas_associadas_ao_operador")))
        .withColumn("data_inicio_relacionamento_banco", limpar_data(F.col("data_inicio_relacionamento_banco")))
        .withColumn("data_abertura_conta",              limpar_data(F.col("data_abertura_conta")))
        .withColumn("data_fundacao_empresa",            limpar_data(F.col("data_fundacao_empresa")))
        .withColumn("data_criacao_operador",            limpar_data(F.col("data_criacao_operador")))
        .withColumn("ref_token",                        limpar_data(F.col("ref_token")))
        .withColumn("ref_anomes", limpar_ref_anomes(F.col("ref_anomes")))
    )


# Executando a limpeza 
df_clean = aplicar_limpeza(df_raw)

# Verificando se existem registros duplicados na base
# Contando todas as linhas depois da limpeza
total_registros = df_clean.count()
# Contando todas as linhas únicas depois da limpeza
total_registros_unicos = (
    df_clean
    .dropDuplicates()
    .count()
)
# Faendo a diferença para encontrar o número de registros duplicados
duplicados_encontrados = (
    total_registros
    - total_registros_unicos
)

print(
    f"Registros duplicados encontrados: "
    f"{duplicados_encontrados}"
)

# Removendo apenas registros iguais em todas as colunas
df_dedup = df_clean.dropDuplicates()

linhas_removidas_dedup = (
    total_registros
    - df_dedup.count()
)

print(
    f"Limpeza concluída. "
    f"{linhas_removidas_dedup} linhas duplicadas removidas."
)



Registros duplicados encontrados: 0
Limpeza concluída. 0 linhas duplicadas removidas.


## 4. Transformações de Negócio

Nesta etapa apliquei as regras de negócio solicitadas no case para gerar os indicadores finais.

As classificações foram criadas a partir das informações já existentes na base após o processo de limpeza e padronização.

In [20]:
def transformacoes_negocio(df: DataFrame) -> DataFrame:
    # Tarefa 2: Faixa Tempo Relacionamento
    dias = F.datediff(F.current_date(), F.col("data_inicio_relacionamento_banco"))
    df = df.withColumn(
        "faixatemporelacionamento",
        F.when(F.col("data_inicio_relacionamento_banco").isNull(), None)
         .when(dias <=  180, "1. Até 6 meses")
         .when(dias <=  365, "2. Entre 6 meses e 1 ano")
         .when(dias <= 1095, "3. Entre 1 e 3 anos")
         .when(dias <= 1825, "4. Entre 3 e 5 anos")
         .when(dias <= 3650, "5. Entre 5 e 10 anos")
         .otherwise("6. Mais de 10 anos")
    )

    # Tarefa 3: Score Maturidade Digital
    def pontuar(flag, pontos):
        return F.when(F.col(flag) == "sim", pontos).otherwise(0)
    
    score = (
        pontuar("tem_token_mobile_habilitado",     3)
      + pontuar("tem_token_embarcado_habilitado",  2)
      + pontuar("flag_acessou_mobile",             2)
      + pontuar("flag_acessou_web",                1)
      + pontuar("flag_acessou_app_comp",           1)
      + pontuar("flag_acessou_canal_90d_anteriores", 1)
      + F.when(F.coalesce(F.col("qtd_total_sessoes"), F.lit(0)) > 10, 1).otherwise(0)
    )
    df = (df.withColumn("scorematuridadedigital", score)
            .withColumn("classificacao_digital",
                F.when(F.col("scorematuridadedigital") <= 2, "Baixo")
                 .when(F.col("scorematuridadedigital") <= 5, "Médio")
                 .when(F.col("scorematuridadedigital") <= 8, "Alto")
                 .otherwise("Avançado")))

    # Tarefa 4: Risco Operador
    tem_mobile = F.col("tem_token_mobile_habilitado") == "sim"
    tem_embarcado = F.col("tem_token_embarcado_habilitado") == "sim"
    sem_token = (~tem_mobile) & (~tem_embarcado)
    apenas_um_token = (tem_mobile & ~tem_embarcado) | (~tem_mobile & tem_embarcado)
    qtd_contas = F.coalesce(F.col("qtd_contas_associadas_ao_operador"), F.lit(0))

    risco = (
        F.when(F.col("sit_operador") == "bloqueado", "ALTO")
         .when(sem_token & (F.col("flag_firmas_e_poderes") == "sim"), "ALTO")
         .when(qtd_contas > 10, "ALTO")
         .when((F.col("sit_operador") == "ativo") & apenas_um_token, "MÉDIO")
         .when(F.col("flag_acessou_canal_90d_anteriores") == "nao", "MÉDIO")
         .otherwise("BAIXO")
    )
    df = df.withColumn("risco_operador", risco)

    # Tarefa 5: Concentração de Operadores
    qtd_op = F.coalesce(F.col("qtd_operadores_associados_ao_cnpj"), F.lit(0))
    df = (df.withColumn("concentracao_operadores",
                F.when(qtd_op == 1, "Operador Único")
                 .when(qtd_op.between(2, 3), "Baixa Concentração")
                 .when(qtd_op.between(4, 10), "Média Concentração")
                 .when(qtd_op > 10, "Alta Concentração")
                 .otherwise(None))
            .withColumn("flagoperadormulticontas",
                F.when(qtd_contas > 1, "sim").otherwise("nao")))

    return df

df_t = transformacoes_negocio(df_dedup)

## 5. Métricas de Observabilidade

Após as transformações, gerei algumas métricas para acompanhar o resultado do processamento.

As métricas ajudam a entender:

- Quantidade de registros processados
- Possíveis perdas de registros
- Problemas de qualidade encontrados
- Distribuição geral dos dados processados

In [21]:
# Coletando métricas baseadas no dataframe processado
dq = df_t.agg(
    F.sum(F.when(F.col("cnpj14").isNull() | (F.length("cnpj14") != 14), 1).otherwise(0)).alias("cnpj_invalido"),
    F.sum(F.when(F.col("ref_anomes").isNull(), 1).otherwise(0)).alias("ref_anomes_nulo"),
    F.avg("scorematuridadedigital").alias("score_medio")
).collect()[0].asDict()

total_in = linhas_brutas
total_out = df_t.count()

metricas = {
    "exec_id": exec_id,
    "exec_ts": datetime.now().isoformat(timespec="seconds"),
    "linhas_in": total_in,
    "linhas_out": total_out,
    "perda_pct": round(100.0 * (total_in - total_out) / max(total_in, 1), 2),
    "cnpj_invalido": int(dq["cnpj_invalido"]) if dq["cnpj_invalido"] else 0,
    "ref_anomes_nulo": int(dq["ref_anomes_nulo"]) if dq["ref_anomes_nulo"] else 0,
    "score_medio": float(dq["score_medio"]) if dq["score_medio"] else 0
}

print("=" * 40)
print(f"RELATORIO DE OBSERVABILIDADE")
print("=" * 40)
for k, v in metricas.items():
    print(f"  {k:20s} : {v}")

# DATA QUALITY GATES
gates = {
    "cnpj_sem_invalidos": metricas["cnpj_invalido"] == 0,
    "ref_anomes_nao_nulo": metricas["ref_anomes_nulo"] == 0,
    "perda_aceitavel": metricas["perda_pct"] < 20,
}

print("\n--- DATA QUALITY GATES ---")
for nome, passou in gates.items():
    status = "PASS" if passou else "FAIL"
    print(f"  [{status}] {nome}")

RELATORIO DE OBSERVABILIDADE
  exec_id              : 20260609005726
  exec_ts              : 2026-06-09T00:57:32
  linhas_in            : 5000
  linhas_out           : 5000
  perda_pct            : 0.0
  cnpj_invalido        : 263
  ref_anomes_nulo      : 1188
  score_medio          : 5.2612

--- DATA QUALITY GATES ---
  [FAIL] cnpj_sem_invalidos
  [FAIL] ref_anomes_nao_nulo
  [PASS] perda_aceitavel


## 6. Salvamento

Nesta etapa preparei a estrutura final da base, incluindo informações de processamento como data de execução e identificador da carga.

O resultado foi salvo em formato Parquet para facilitar consultas e reutilização dos dados em etapas futuras.

Durante os testes locais utilizei a gravação via Pandas para garantir a execução completa do pipeline.

In [22]:
output_path = "saida_parquet"
metrics_path = "saida_metrics"

df_final = (
    df_t
    .withColumn(
        "scorematuridadedigital",
        F.col("scorematuridadedigital").cast(IntegerType())
    )
    # Mantido como string apenas para compatibilidade do teste local
    .withColumn(
        "ref_anomes",
        F.col("ref_anomes").cast("string")
    )
    .withColumn(
        "dt_processamento",
        F.current_timestamp().cast("string")
    )
    .withColumn(
        "exec_id",
        F.lit(exec_id)
    )
)

print("\nSalvando camada Refined...")

# Salvamento utilizado durante os testes locais
df_pandas = df_final.toPandas()
df_pandas.to_parquet(
    f"{output_path}_notebook.parquet"
)

pd.DataFrame([metricas]).to_parquet(
    f"{metrics_path}_notebook.parquet"
)

print(
    f"Sucesso! Base refinada salva em: "
    f"{output_path}_notebook.parquet"
)

print(
    f"Sucesso! Métricas salvas em: "
    f"{metrics_path}_notebook.parquet"
)


Salvando camada Refined...
Sucesso! Base refinada salva em: saida_parquet_notebook.parquet
Sucesso! Métricas salvas em: saida_metrics_notebook.parquet


## 7. Validação da Saída

Após a gravação da camada final, realizei uma leitura do arquivo gerado para validar se os dados foram salvos corretamente.

Também exibi algumas colunas importantes do resultado final para confirmar a aplicação das regras de negócio implementadas no pipeline.

In [33]:
import pandas as pd

# Caminho do arquivo gerado na etapa anterior
caminho_saida = "saida_parquet_notebook.parquet"

# Leitura do arquivo Parquet final
df_validacao = pd.read_parquet(caminho_saida)

print(f"Leitura concluída com sucesso!")
print(f"Total de registos no arquivo: {df_validacao.shape[0]}")
print(f"Quantidade de colunas na camada Refined: {df_validacao.shape[1]}\n")

# Selecionamos as colunas-chave do case para demonstrar o resultado final
colunas_exibicao = [
    "cnpj14", 
    "cod_operador", 
    "risco_operador", 
    "scorematuridadedigital", 
    "classificacao_digital"
]

print("Amostra dos principais dados processados (Camada Refined):")
display(df_validacao[colunas_exibicao].head(5))



Leitura concluída com sucesso!
Total de registos no arquivo: 5000
Quantidade de colunas na camada Refined: 46

Amostra dos principais dados processados (Camada Refined):


,cnpj14,cod_operador,risco_operador,scorematuridadedigital,classificacao_digital
0,24692911686986,1004,BAIXO,8,Alto
1,38731170462929,1004,MÉDIO,6,Alto
2,33360031674251,1001,MÉDIO,4,Médio
3,50024622304981,1002,BAIXO,4,Médio
4,87114204788800,1002,BAIXO,4,Médio


In [35]:
# Validacao completa da camada Refined 
print("Amostra completa da camada Refined:")
display(df_validacao.head(10))

Amostra completa da camada Refined:


,cnpj14,grupo_segmento,segmento_detalhado,modelo_atendimento,situacao_conta,situacao_cadastral_receita,data_abertura_conta,faixa_tempo_criacao_conta,data_fundacao_empresa,faixa_tempo_vida_empresa,...,qtd_sessoes_web,ref_anomes,faixatemporelacionamento,scorematuridadedigital,classificacao_digital,risco_operador,concentracao_operadores,flagoperadormulticontas,dt_processamento,exec_id
0,24692911686986,PRIVATE,Atacado Middle,agencia,ativa,INAPTA,2023-09-27,Mais de 3 anos,2004-12-20,Mais de 5 anos,...,5,NaN,6. Mais de 10 anos,8,Alto,BAIXO,Operador Único,sim,2026-06-09 00:57:32.237125,20260609005726
1,38731170462929,ATACADO,Varejo PJ,agencia,encerrada,INAPTA,2014-03-01,1 a 3 anos,1991-02-03,Ate 5 anos,...,5,2026-05-01,5. Entre 5 e 10 anos,6,Alto,MÉDIO,NaN,sim,2026-06-09 00:57:32.237125,20260609005726
2,33360031674251,ATACADO,null,agencia,bloqueada,INAPTA,2014-11-06,Ate 1 ano,2012-03-07,Mais de 5 anos,...,0,NaN,6. Mais de 10 anos,4,Médio,MÉDIO,Alta Concentração,sim,2026-06-09 00:57:32.237125,20260609005726
3,50024622304981,VAREJO,null,NaN,ativa,ATIVA,2012-02-24,Ate 1 ano,2009-04-11,Mais de 5 anos,...,5,2026-05-01,6. Mais de 10 anos,4,Médio,BAIXO,Baixa Concentração,sim,2026-06-09 00:57:32.237125,20260609005726
4,87114204788800,ATACADO,null,digital,bloqueada,ATIVA,2015-09-19,Mais de 3 anos,2011-01-02,Mais de 5 anos,...,5,2026-05-01,6. Mais de 10 anos,4,Médio,BAIXO,Baixa Concentração,nao,2026-06-09 00:57:32.237125,20260609005726
5,69837863492303,VAREJO,Varejo PJ,agencia,bloqueada,INAPTA,2015-10-22,Ate 1 ano,1992-12-15,Ate 5 anos,...,0,2026-05-01,5. Entre 5 e 10 anos,5,Médio,MÉDIO,Baixa Concentração,sim,2026-06-09 00:57:32.237125,20260609005726
6,90852097848207,ATACADO,Atacado Middle,digital,ativa,INAPTA,2023-12-29,Mais de 3 anos,2000-07-21,Ate 5 anos,...,0,2026-05-01,6. Mais de 10 anos,5,Médio,MÉDIO,NaN,sim,2026-06-09 00:57:32.237125,20260609005726
7,53904199666098,VAREJO,Varejo PJ,digital,ativa,INAPTA,2023-01-31,Mais de 3 anos,2014-11-19,Ate 5 anos,...,0,2026-05-01,5. Entre 5 e 10 anos,7,Alto,ALTO,NaN,sim,2026-06-09 00:57:32.237125,20260609005726
8,39715723428965,VAREJO,null,digital,ativa,INAPTA,2013-09-09,Mais de 3 anos,2021-12-25,Mais de 5 anos,...,0,2026-05-01,4. Entre 3 e 5 anos,5,Médio,BAIXO,Alta Concentração,nao,2026-06-09 00:57:32.237125,20260609005726
9,65126861753173,ATACADO,Atacado Middle,agencia,bloqueada,ATIVA,2020-02-13,Mais de 3 anos,2012-08-23,Mais de 5 anos,...,5,NaN,6. Mais de 10 anos,5,Médio,MÉDIO,Operador Único,sim,2026-06-09 00:57:32.237125,20260609005726
